# Building a Trustworthy Housing-Price Baseline

**Prediction question:** How might an analyst estimate a property's eventual recorded price using attributes available before that price is known? The table is an adapted educational case, not verified live business records. Begin with `S18_Housing_Workflow.csv` and consult `S18_Data_Notes.md` for field meanings and source limits.

In this practical you will audit the rows, protect training/validation/test roles, handle imperfect inputs through a repeatable path, fit a simple constant reference, compare two validation errors and explain what the comparison cannot show. You need a Python 3 Jupyter kernel with pandas, NumPy and scikit-learn. Run cells in order. Cells marked **Your turn** need code or a written response before their dependent checkpoint can run; they are intentionally unfinished.

**Local setup:** use Python 3.12 with `python -m pip install -r S18_requirements.txt`, then select that environment's Python kernel in your Jupyter frontend. Start Jupyter with this folder as the working folder. If using a different working folder, edit `DATA_PATH` in the load cell. In a hosted notebook, upload the CSV and point `DATA_PATH` to it; that environment has not been tested here.

In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| NumPy", np.__version__, "| scikit-learn", sklearn.__version__)

## 1. What would you predict?

One row is a recorded observation. `price` is the target; `X` will contain attributes available before the price is known. The added `record_id` is for audit, and `price_per_area_at_sale` was calculated from the outcome. The source does not establish currency, area units, location or collection date.

**Your turn — explanation:** Write one sentence naming the prediction moment, target and intended use of an initial reference. Why must `price_per_area_at_sale` be excluded even if it looks predictive? Why is `record_id` also excluded?

*Response:* …

In [ ]:
NUMERIC = ["area", "bedrooms", "bathrooms", "stories", "parking"]
CATEGORICAL = ["mainroad", "guestroom", "basement", "hotwaterheating",
               "airconditioning", "prefarea", "furnishingstatus"]
RAW_FEATURES = NUMERIC + CATEGORICAL
NUMERIC_WITH_RATIO = NUMERIC + ["area_per_bedroom"]
EXPECTED_COLUMNS = [
    "price", "area", "bedrooms", "bathrooms", "stories", "mainroad",
    "guestroom", "basement", "hotwaterheating", "airconditioning",
    "parking", "prefarea", "furnishingstatus", "record_id",
    "price_per_area_at_sale",
]
FORBIDDEN_INPUTS = {"price", "record_id", "price_per_area_at_sale"}
assert len(RAW_FEATURES) == 12 and not FORBIDDEN_INPUTS.intersection(RAW_FEATURES)

The supplied field lists are the raw prediction contract: five numerical and seven categorical attributes. `area_per_bedroom` will be derived **inside** the repeated preparation path, not read as an input column. On a fresh property row, only the twelve raw feature fields are required.

## 2. Load and audit identity

The supplied load cell keeps all raw CSV tokens as strings so that blank values and malformed tokens remain visible. If the file is not found, put the CSV beside this notebook in the working folder or edit `DATA_PATH`.

In [ ]:
# Keep this notebook and S18_Housing_Workflow.csv in the same working folder.
# If a Jupyter service starts elsewhere, edit only this path.
DATA_PATH = Path("S18_Housing_Workflow.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH.resolve()}. Put the approved teaching CSV beside "
        "this notebook or set DATA_PATH to its local path."
    )

# Read every token as text so blank cells, comma-grouped area and count suffixes
# remain distinguishable until the fixed input-cleaning rule runs.
raw = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
assert raw.columns.tolist() == EXPECTED_COLUMNS, "Load checkpoint: unexpected columns or order"
assert raw.shape == (553, 15), "Load checkpoint: expected 553 rows and 15 columns"
assert raw["price"].ne("").all(), "Load checkpoint: a missing target cannot be an observed label"
print("Loaded rows/columns:", raw.shape)
print("Raw feature types:", raw[RAW_FEATURES].dtypes.value_counts().to_dict())

**Your turn — code and explanation:** Inspect repeated `record_id` values. Verify that each repeated-ID group consists of identical full rows before keeping one copy. Build `dedup`, `X`, `y` and `ids` with aligned indexes. Explain why a repeated teaching ID is not proof of a real-world duplicate property.

**Checkpoint:** expect 545 retained observations, eight removed copies, `X` with 12 columns and no conflicting repeated ID. If copies conflict, stop and investigate rather than choosing an arbitrary row. Complete this cell before splitting.

In [ ]:
# YOUR TURN: inspect repeated IDs, reject conflicting copies, then define
# dedup, X, y and ids. Use RAW_FEATURES and keep price out of X.
# Hint: value_counts(), drop_duplicates(), pd.to_numeric() and index.equals().

## 3. Reserve independent evaluation

The source rows are ordered by decreasing price, so do not split by position. First reserve 20% as test. Then take 25% of the remaining rows for validation. Keep `X`, `y` and `ids` together in both `train_test_split` calls; use `shuffle=True, random_state=42` each time. Do not inspect reserved test targets or fit any preparation rule before this step.

**Your turn — code:** Create `X_train`, `X_val`, `X_test`; aligned `y_train`, `y_val`, `y_test`; and `id_train`, `id_val`, `id_test`. Check counts, ID disjointness and index alignment. The test portion is set aside; it is not scored in this session.

**Checkpoint:** 327 training, 109 validation and 109 reserved test IDs, with zero overlap. Explain in one sentence what each partition may influence. Complete this cell before inspecting training quality.

In [ ]:
# YOUR TURN: make both prescribed shuffled splits and check alignment.
# Hint: train_test_split accepts X, y and ids in one call and returns pairs
# in the same order. Use test_size=0.20, then test_size=0.25.

## 4. Judge training-data quality

Work from **training inputs** for quality decisions. A blank is missing; a present token may need controlled parsing. Zero parking is valid. Nonpositive area is invalid, while a large positive area is a candidate for inspection rather than automatic deletion.

**Your turn — inspection:** Count blanks in `area`, `bathrooms`, `mainroad` and `furnishingstatus`; list unusual area/bathroom tokens and raw categorical variants in training. Record a short action and reason for each issue type. Do not drop every row with an issue.

**Checkpoint:** your inspection should reveal both missingness and parseable tokens in training. What additional evidence would you need before deleting an unusually large positive area?

In [ ]:
# YOUR TURN: inspect X_train only. Print compact counts or small lists,
# not the reserved test rows. A supplied worked check follows in class.

## 5. Use a transparent fixed input rule

The next cell is **provided** because implementing a robust row-level parser from scratch would crowd out the new ML ideas. Read it before running it. Identify where it (a) permits only known numeric formats, (b) corrects known category variants while preserving a new furnishing term, (c) turns impossible area/bedroom values into missing, and (d) builds `area_per_bedroom` without price. It never learns a median, mode, scale or category vocabulary from other rows.

In [ ]:
def parse_whole_number(value, field):
    """Parse only declared integer forms. Blank -> missing; unknown token -> error."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        if not np.isfinite(value) or not float(value).is_integer():
            raise ValueError(f"{field}: expected a finite whole number, got {value!r}")
        return float(value)
    token = str(value).strip().lower()
    if token == "":
        return np.nan
    if field == "area" and re.fullmatch(r"\d{1,3}(?:,\d{3})+", token):
        token = token.replace(",", "")
    if field == "bathrooms" and re.fullmatch(r"\d+ baths?", token):
        token = token.split()[0]
    if not re.fullmatch(r"[+-]?\d+", token):
        raise ValueError(f"{field}: unrecognized numeric token {value!r}")
    return float(int(token))


def clean_category(value, field):
    """Apply fixed known corrections; preserve a well-formed unseen furnishing term."""
    if pd.isna(value):
        return np.nan
    token = str(value).strip().lower()
    if token == "":
        return np.nan
    if field == "furnishingstatus":
        token = {"semi furnished": "semi-furnished",
                 "unfurnshed": "unfurnished"}.get(token, token)
        if not re.fullmatch(r"[a-z]+(?:-[a-z]+)*", token):
            raise ValueError(f"{field}: unrecognized category format {value!r}")
        return token  # A new well-formed term stays new for the encoder.
    if token not in {"yes", "no"}:
        raise ValueError(f"{field}: expected yes/no or missing, got {value!r}")
    return token


def clean_features(frame):
    """Row-preserving, target-independent rules for the twelve raw inputs."""
    if not isinstance(frame, pd.DataFrame) or frame.columns.tolist() != RAW_FEATURES:
        raise ValueError("Expected a DataFrame with exactly the twelve ordered raw feature columns")
    cleaned = frame.copy()
    for field in NUMERIC:
        cleaned[field] = cleaned[field].map(
            lambda value, field=field: parse_whole_number(value, field)
        ).astype(float)
    # These are fixed domain rules, not statistics learned from other rows.
    for field in ["area", "bedrooms", "bathrooms", "stories"]:
        cleaned.loc[cleaned[field] <= 0, field] = np.nan
    cleaned.loc[cleaned["parking"] < 0, "parking"] = np.nan  # zero is valid
    for field in CATEGORICAL:
        cleaned[field] = cleaned[field].map(
            lambda value, field=field: clean_category(value, field)
        )
    cleaned["area_per_bedroom"] = cleaned["area"] / cleaned["bedrooms"]
    cleaned.loc[~np.isfinite(cleaned["area_per_bedroom"]), "area_per_bedroom"] = np.nan
    assert len(cleaned) == len(frame) and cleaned.index.equals(frame.index)
    return cleaned


def names_with_ratio(transformer, input_features):
    return np.asarray(list(input_features) + ["area_per_bedroom"], dtype=object)

**Your turn — behavior check:** Apply `clean_features` to `X_train`. Check that row count and index stay fixed, `parking=0` remains valid and invalid/missing area yields a missing ratio. Try a one-row copy with `bedrooms=0`. Try an unfamiliar area token such as `about 4000`; it should raise a clear error rather than disappear silently.

In [ ]:
# YOUR TURN: define clean_train and test the fixed helper's behavior.
# Hint: copy one training row before changing a probe value.

**Your turn — training-only unusual-value check:** Use the cleaned **training** area values to calculate Q1, Q3 and a 1.5×IQR upper flag. Count flagged positive areas. Do not remove them just for being flagged. Explain how an invalid nonpositive area differs from a large positive one.

In [ ]:
# YOUR TURN: compute a training-only IQR flag for recorded area.

## 6. Build and inspect fitted mixed-data preparation

Follow the flow: twelve raw features → fixed row-level cleaning and safe ratio → numerical median imputation then scaling; categorical most-frequent imputation then one-hot encoding → constant baseline. `fit` learns medians, modes, scales and encoder categories **from training only**. The numerical branch below is provided as the live starting point.

In [ ]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

**Your turn — code:** Build `categorical_pipe` with `SimpleImputer(strategy="most_frequent")` followed by `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`. Then combine `numeric_pipe` and `categorical_pipe` in `columns = ColumnTransformer(...)` using the explicit lists `NUMERIC_WITH_RATIO` and `CATEGORICAL`, with `remainder="drop"`. Assemble `median_model = Pipeline(...)` with `FunctionTransformer(clean_features, validate=False, feature_names_out=names_with_ratio)`, then `columns`, then `DummyRegressor(strategy="median")`. Fit it on `X_train, y_train` only.

**Checkpoint:** the fitted object must accept the original twelve-column raw input. If the setup fails, compare each branch's step order and exact column list with the flow above; a trainer can provide a recovery checkpoint after your attempt. Complete this cell before inspection or scoring.

In [ ]:
# YOUR TURN: define categorical_pipe, columns and median_model, then fit.
# Keep the step names clean, columns and baseline so later checks can find them.

**Your turn — independent preparation evidence:** Use `median_model[:-1].transform(X_train)` and `.transform(X_val)` to inspect row counts, transformed width, finite values and feature names. Find the fitted numeric `statistics_`, encoder `categories_` and scale mean. Verify they remain unchanged after transforming validation. Explain why a constant baseline's MAE cannot prove these feature transformations are useful.

**Checkpoint:** no missing/nonfinite transformed values, same width and names for training/validation, unchanged fitted statistics. Get the width from `get_feature_names_out()` rather than guessing it.

In [ ]:
# YOUR TURN: inspect the fitted branches and transformed matrices.
# Hint: median_model.named_steps["columns"].named_transformers_["num"]
# and ...["cat"] reveal fitted branch steps.

## 7. Establish a median reference and use a fresh raw row

**Your turn — code and interpretation:** Predict on validation with `median_model`, calculate `mean_absolute_error(y_val, predictions)` and compare it with your own mean of absolute differences. Check that every prediction equals the **training-target median**. State the MAE in price units, not as an accuracy percentage.

In [ ]:
# YOUR TURN: define median_predictions and median_mae, check the constant
# against np.median(y_train), and explain the result in a sentence below.

*Median reference interpretation:* …

**Your turn — fresh input:** Make a separate **synthetic** DataFrame with only `RAW_FEATURES`, one missing eligible value and a well-formed furnishing term unseen in training (for example, `part-furnished`). Pass it through the fitted pipeline. Check that its furnishing one-hot block is all zeros, not just that `predict` returns a number. This fallback does not prove the new category has a useful meaning to the model.

In [ ]:
# YOUR TURN: create fresh with exactly RAW_FEATURES, predict with median_model,
# then inspect the furnishing block of the fitted categorical encoder.
# Do not put a synthetic row into the reserved test partition.

## 8. Independent application: mean reference and fair choice

**Your turn — protected attempt:** Create a **new unfitted** candidate for `DummyRegressor(strategy="mean")` from the demonstrated structure. A `clone(median_model)` is one way to get independent unfitted steps; set its `baseline__strategy` to `"mean"`. Fit on the same training rows, predict the same validation rows and calculate validation MAE. Confirm its predictions equal the training-target mean. Choose the lower validation MAE, with median as the tie-breaker. Do not open the reserved test labels to settle the comparison.

Explain why two property rows with different attributes receive the same prediction from either baseline. Defend `area_per_bedroom` as available before price is known and reject `price_per_area_at_sale` as unavailable at that moment.

In [ ]:
# YOUR TURN: define and fit mean_model independently; compute mean_mae;
# compare the two validation MAEs using the stated tie rule.

*My development reference and reason:* …

*Why these two constants ignore property differences:* …

*Why the two area ratios have different eligibility:* …

## 9. What can this workflow support?

**Your turn — synthesis:** Name one verified property of the data split, one training-fitted quantity, and one claim the current validation comparison cannot support. The test set remains reserved.

Cross-validation is a way to obtain more development evidence than one validation split: within development data, rotate `A+B → C`, `A+C → B`, `B+C → A`, fitting the **whole** preprocessing-and-estimator pipeline anew on each fold's training portion. Keep the reserved test separate. Why would preparing all rows once before rotating folds be unsafe? No cross-validation code is required here.

*My explanation:* …

**Completion check:** Can your fitted object accept a twelve-feature raw row, transform it to finite values, make a constant prediction and compare both baselines by validation MAE without using the test labels? If a checkpoint fails, repair the earlier dependency before interpreting the result.